# Task 5 — Pivot Translation: Spanish → English → Swedish

## 1. Setup


In [ ]:
from __future__ import annotations

import copy
import json
import random
import re
import time
from collections import Counter
from functools import partial
from itertools import zip_longest
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from gensim.models import Word2Vec
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.chrf_score import corpus_chrf
from torch import nn
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


SEED = 43
DEVICE = torch.device("cpu")

SAMPLE_FRACTION = 0.10
MAX_WORD_TOKENS = 50
MAX_CHARACTERS = 220
MAX_VOCAB_SIZE = 20_000
MIN_WORD_FREQUENCY = 2

BATCH_SIZE = 96
EVALUATION_BATCH_SIZE = 128
EPOCHS = 5
PATIENCE = 2
MAX_GENERATION_LENGTH = MAX_WORD_TOKENS + 10

CONFIG = {
    "embedding_dim": 128,
    "hidden_size": 192,
    "dropout": 0.20,
    "learning_rate": 0.001,
}

FORCE_REBUILD_SAMPLE = False
FORCE_RETRAIN = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(8, torch.get_num_threads())))

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data").exists()),
    Path.cwd(),
)

DATA_DIR = PROJECT_ROOT / "data"
PREPROCESSED_DIR = DATA_DIR / "preprocessed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "task05"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RAW_EN_PATH = DATA_DIR / "europarl-v7.sv-en.en"
RAW_SV_PATH = DATA_DIR / "europarl-v7.sv-en.sv"
SAMPLED_EN_PATH = PREPROCESSED_DIR / "europarl_sv_en_10pct_preprocessed.en"
SAMPLED_SV_PATH = PREPROCESSED_DIR / "europarl_sv_en_10pct_preprocessed.sv"
SPANISH_PATH = PREPROCESSED_DIR / "europarl_10pct_preprocessed.es"

TASK4_CHECKPOINT = (
    PROJECT_ROOT / "artifacts" / "task04" / "attention_word_es_to_en.pt"
)
TASK5_CHECKPOINT = ARTIFACT_DIR / "attention_word_en_to_sv.pt"

required = [RAW_EN_PATH, RAW_SV_PATH, SPANISH_PATH, TASK4_CHECKPOINT]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n- " + "\n- ".join(missing))

print("Device:", DEVICE)
print("Task 4 checkpoint:", TASK4_CHECKPOINT)
print("Swedish–English corpus:", RAW_EN_PATH, "and", RAW_SV_PATH)


Device: cpu
Task 4 checkpoint: /home/theohagen/Desktop/nlp-machine-translation/artifacts/task04/attention_word_es_to_en.pt
Swedish–English corpus: /home/theohagen/Desktop/nlp-machine-translation/data/europarl-v7.sv-en.en and /home/theohagen/Desktop/nlp-machine-translation/data/europarl-v7.sv-en.sv


/home/theohagen/miniforge3/envs/nlp-mt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Prepare the Swedish–English data

The same preprocessing as in Task 2 is used.


In [2]:
WORD_PATTERN = re.compile(r"\w+(?:['’]\w+)*|[^\w\s]", flags=re.UNICODE)


def word_tokenize(text):
    return WORD_PATTERN.findall(text)


def detokenize_words(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:%)])", r"\1", text)
    return re.sub(r"([¿¡(])\s+", r"\1", text)


def normalize_sentence(text):
    return " ".join(text.strip().lower().split())


def prepare_sample():
    if (
        SAMPLED_EN_PATH.exists()
        and SAMPLED_SV_PATH.exists()
        and not FORCE_REBUILD_SAMPLE
    ):
        print("Using existing preprocessed 10% sample.")
        return

    with RAW_EN_PATH.open(encoding="utf-8", errors="replace") as file:
        english_lines = sum(1 for _ in file)
    with RAW_SV_PATH.open(encoding="utf-8", errors="replace") as file:
        swedish_lines = sum(1 for _ in file)

    if english_lines != swedish_lines:
        raise ValueError("The Swedish–English corpus files are not aligned.")

    sample_size = int(SAMPLE_FRACTION * english_lines)
    selected = set(
        np.random.default_rng(SEED).choice(
            english_lines,
            size=sample_size,
            replace=False,
        )
    )

    temporary_en = SAMPLED_EN_PATH.with_suffix(".en.tmp")
    temporary_sv = SAMPLED_SV_PATH.with_suffix(".sv.tmp")
    retained = 0

    try:
        with (
            RAW_EN_PATH.open(encoding="utf-8", errors="replace") as en_file,
            RAW_SV_PATH.open(encoding="utf-8", errors="replace") as sv_file,
            temporary_en.open("w", encoding="utf-8") as out_en,
            temporary_sv.open("w", encoding="utf-8") as out_sv,
        ):
            pairs = zip_longest(en_file, sv_file, fillvalue=None)
            for index, (english, swedish) in enumerate(
                tqdm(pairs, total=english_lines, desc="Sampling")
            ):
                if english is None or swedish is None:
                    raise ValueError("The corpus files lost alignment.")
                if index not in selected:
                    continue

                english = english.rstrip("\r\n")
                swedish = swedish.rstrip("\r\n")
                if (
                    not english.strip()
                    or not swedish.strip()
                    or english.lstrip().startswith("<")
                    or swedish.lstrip().startswith("<")
                ):
                    continue

                out_en.write(normalize_sentence(english) + "\n")
                out_sv.write(normalize_sentence(swedish) + "\n")
                retained += 1

        temporary_en.replace(SAMPLED_EN_PATH)
        temporary_sv.replace(SAMPLED_SV_PATH)
    except Exception:
        temporary_en.unlink(missing_ok=True)
        temporary_sv.unlink(missing_ok=True)
        raise

    print(f"Retained {retained:,} of {sample_size:,} sampled pairs.")


prepare_sample()


Sampling: 100%|██████████| 1862234/1862234 [00:01<00:00, 1267936.11it/s]

Retained 184,834 of 186,223 sampled pairs.


## 3. Load, split, and batch


In [3]:
def load_parallel_data():
    rows = []
    with (
        SAMPLED_EN_PATH.open(encoding="utf-8") as en_file,
        SAMPLED_SV_PATH.open(encoding="utf-8") as sv_file,
    ):
        for index, (english, swedish) in enumerate(
            zip_longest(en_file, sv_file, fillvalue=None)
        ):
            if english is None or swedish is None:
                raise ValueError("The preprocessed files are not aligned.")

            english = english.rstrip("\r\n")
            swedish = swedish.rstrip("\r\n")
            en_length = len(word_tokenize(english))
            sv_length = len(word_tokenize(swedish))

            if (
                0 < en_length <= MAX_WORD_TOKENS
                and 0 < sv_length <= MAX_WORD_TOKENS
                and len(english) <= MAX_CHARACTERS
                and len(swedish) <= MAX_CHARACTERS
            ):
                rows.append((index, english, swedish))

    return pd.DataFrame(rows, columns=["original_index", "english", "swedish"])


data = load_parallel_data()
indices = np.random.default_rng(SEED).permutation(len(data))
n_test = int(0.20 * len(data))
n_validation = int(0.10 * len(data))

splits = {
    "test": data.iloc[indices[:n_test]].reset_index(drop=True),
    "validation": data.iloc[
        indices[n_test:n_test + n_validation]
    ].reset_index(drop=True),
    "train": data.iloc[
        indices[n_test + n_validation:]
    ].reset_index(drop=True),
}

display(
    pd.DataFrame(
        {"split": name, "pairs": len(frame)}
        for name, frame in splits.items()
    )
)


PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3


class Vocabulary:
    def __init__(self, texts=None, max_size=None, min_frequency=1, stoi=None):
        if stoi is not None:
            self.itos = [None] * len(stoi)
            for token, index in stoi.items():
                self.itos[index] = token
        else:
            counts = Counter(
                token
                for text in tqdm(texts, desc="Vocabulary")
                for token in word_tokenize(text)
            )
            tokens = [
                token
                for token, frequency in counts.most_common()
                if frequency >= min_frequency
            ]
            if max_size is not None:
                tokens = tokens[:max_size - 4]
            self.itos = [PAD, UNK, BOS, EOS, *tokens]

        self.stoi = {token: index for index, token in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        return [
            BOS_ID,
            *[self.stoi.get(token, UNK_ID) for token in word_tokenize(text)],
            EOS_ID,
        ]

    def decode(self, indices):
        return [
            self.itos[index]
            for index in indices
            if index not in {PAD_ID, BOS_ID, EOS_ID}
        ]


class TranslationDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return row["english"], row["swedish"], row["original_index"]


def collate_batch(batch, source_vocab, target_vocab):
    sources, targets, original_indices = zip(*batch)
    source_sequences = [
        torch.tensor(source_vocab.encode(text), dtype=torch.long)
        for text in sources
    ]
    target_sequences = [
        torch.tensor(target_vocab.encode(text), dtype=torch.long)
        for text in targets
    ]
    return {
        "source": pad_sequence(
            source_sequences, batch_first=True, padding_value=PAD_ID
        ),
        "source_lengths": torch.tensor([len(sequence) for sequence in source_sequences]),
        "target": pad_sequence(
            target_sequences, batch_first=True, padding_value=PAD_ID
        ),
        "source_text": sources,
        "target_text": targets,
        "original_index": original_indices,
    }


def make_loader(frame, source_vocab, target_vocab, batch_size, shuffle):
    return DataLoader(
        TranslationDataset(frame),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=partial(
            collate_batch,
            source_vocab=source_vocab,
            target_vocab=target_vocab,
        ),
    )


english_vocab = Vocabulary(
    splits["train"]["english"],
    MAX_VOCAB_SIZE,
    MIN_WORD_FREQUENCY,
)
swedish_vocab = Vocabulary(
    splits["train"]["swedish"],
    MAX_VOCAB_SIZE,
    MIN_WORD_FREQUENCY,
)

train_loader = make_loader(
    splits["train"], english_vocab, swedish_vocab, BATCH_SIZE, True
)
validation_loader = make_loader(
    splits["validation"],
    english_vocab,
    swedish_vocab,
    EVALUATION_BATCH_SIZE,
    False,
)
test_loader = make_loader(
    splits["test"],
    english_vocab,
    swedish_vocab,
    EVALUATION_BATCH_SIZE,
    False,
)

print("English vocabulary:", len(english_vocab))
print("Swedish vocabulary:", len(swedish_vocab))


,split,pairs
0,test,29582
1,validation,14791
2,train,103540


Vocabulary: 100%|██████████| 103540/103540 [00:00<00:00, 222832.83it/s]

English vocabulary: 18325
Swedish vocabulary: 20000


## 4. Attention model

In [4]:
class Seq2SeqGRUAttention(nn.Module):
    def __init__(
        self,
        source_vocab_size,
        target_vocab_size,
        embedding_dim,
        hidden_size,
        dropout,
        source_embeddings=None,
        target_embeddings=None,
    ):
        super().__init__()
        self.source_embedding = nn.Embedding(
            source_vocab_size, embedding_dim, padding_idx=PAD_ID
        )
        self.target_embedding = nn.Embedding(
            target_vocab_size, embedding_dim, padding_idx=PAD_ID
        )
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.decoder = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.attention_combine = nn.Linear(2 * hidden_size, hidden_size)
        self.output = nn.Linear(hidden_size, target_vocab_size)

        if source_embeddings is not None:
            self.source_embedding.weight.data.copy_(
                torch.tensor(source_embeddings)
            )
        if target_embeddings is not None:
            self.target_embedding.weight.data.copy_(
                torch.tensor(target_embeddings)
            )

    def encode(self, source, source_lengths):
        embedded = self.dropout(self.source_embedding(source))
        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, hidden = self.encoder(packed)
        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=source.size(1),
        )
        return encoder_outputs, hidden, source.ne(PAD_ID)

    def decode(self, target_input, hidden, encoder_outputs, source_mask):
        embedded = self.dropout(self.target_embedding(target_input))
        decoder_outputs, hidden = self.decoder(embedded, hidden)

        scores = torch.bmm(
            decoder_outputs,
            encoder_outputs.transpose(1, 2),
        ).masked_fill(~source_mask.unsqueeze(1), float("-inf"))
        attention = torch.softmax(scores, dim=-1)
        context = torch.bmm(attention, encoder_outputs)

        combined = torch.tanh(
            self.attention_combine(
                torch.cat([decoder_outputs, context], dim=-1)
            )
        )
        return self.output(combined), hidden, attention

    def forward(self, source, source_lengths, target_input):
        encoder_outputs, hidden, source_mask = self.encode(
            source, source_lengths
        )
        logits, _, _ = self.decode(
            target_input,
            hidden,
            encoder_outputs,
            source_mask,
        )
        return logits


def build_model(source_vocab, target_vocab, config, source_matrix=None, target_matrix=None):
    return Seq2SeqGRUAttention(
        len(source_vocab),
        len(target_vocab),
        config["embedding_dim"],
        config["hidden_size"],
        config["dropout"],
        source_matrix,
        target_matrix,
    ).to(DEVICE)


## 5. Shared training, evaluation, and checkpoint utilities


In [5]:
def loss_epoch(model, loader, criterion, optimizer=None):
    model.train(optimizer is not None)
    total_loss = 0.0
    total_tokens = 0

    for batch in tqdm(loader, leave=False):
        source = batch["source"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        if optimizer is not None:
            optimizer.zero_grad()

        with torch.set_grad_enabled(optimizer is not None):
            logits = model(
                source,
                batch["source_lengths"],
                target[:, :-1],
            )
            expected = target[:, 1:]
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                expected.reshape(-1),
            )

            if optimizer is not None:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        tokens = expected.ne(PAD_ID).sum().item()
        total_loss += loss.item() * tokens
        total_tokens += tokens

    return total_loss / total_tokens


def fit(model):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["learning_rate"],
    )
    best_state = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    waiting = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        train_loss = loss_epoch(model, train_loader, criterion, optimizer)
        validation_loss = loss_epoch(model, validation_loader, criterion)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": validation_loss,
            }
        )
        print(
            f"epoch={epoch} train_loss={train_loss:.4f} "
            f"validation_loss={validation_loss:.4f}"
        )

        if validation_loss < best_loss:
            best_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            waiting = 0
        else:
            waiting += 1
            if waiting >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return pd.DataFrame(history)


@torch.no_grad()
def translate_batch(model, source, lengths, target_vocab):
    model.eval()
    encoder_outputs, hidden, source_mask = model.encode(source, lengths)
    current = torch.full(
        (source.size(0), 1),
        BOS_ID,
        dtype=torch.long,
        device=DEVICE,
    )
    predictions = [[] for _ in range(source.size(0))]
    finished = torch.zeros(source.size(0), dtype=torch.bool, device=DEVICE)

    for _ in range(MAX_GENERATION_LENGTH):
        logits, hidden, _ = model.decode(
            current,
            hidden,
            encoder_outputs,
            source_mask,
        )
        current = logits[:, -1].argmax(-1, keepdim=True)

        for row, token in enumerate(current.squeeze(1).tolist()):
            if not finished[row]:
                if token == EOS_ID:
                    finished[row] = True
                else:
                    predictions[row].append(token)
        if finished.all():
            break

    return [
        detokenize_words(target_vocab.decode(indices))
        for indices in predictions
    ]


def corpus_metrics(references, hypotheses):
    bleu = 100 * corpus_bleu(
        [[word_tokenize(reference)] for reference in references],
        [word_tokenize(hypothesis) for hypothesis in hypotheses],
        smoothing_function=SmoothingFunction().method1,
    )
    chrf = 100 * corpus_chrf(
        [[reference] for reference in references],
        hypotheses,
    )
    return bleu, chrf


@torch.no_grad()
def evaluate(model, loader, target_vocab):
    rows = []
    for batch in tqdm(loader, desc="Translating", leave=False):
        hypotheses = translate_batch(
            model,
            batch["source"].to(DEVICE),
            batch["source_lengths"],
            target_vocab,
        )
        rows.extend(
            {
                "original_index": index,
                "source": source,
                "reference": reference,
                "hypothesis": hypothesis,
            }
            for index, source, reference, hypothesis in zip(
                batch["original_index"],
                batch["source_text"],
                batch["target_text"],
                hypotheses,
            )
        )

    predictions = pd.DataFrame(rows)
    bleu, chrf = corpus_metrics(
        predictions["reference"].tolist(),
        predictions["hypothesis"].tolist(),
    )
    return bleu, chrf, predictions


@torch.no_grad()
def translate_texts(model, texts, source_vocab, target_vocab, batch_size=128):
    texts = list(texts)
    translations = []

    for start in range(0, len(texts), batch_size):
        current_texts = texts[start:start + batch_size]
        sequences = [
            torch.tensor(source_vocab.encode(text), dtype=torch.long)
            for text in current_texts
        ]
        lengths = torch.tensor([len(sequence) for sequence in sequences])
        source = pad_sequence(
            sequences,
            batch_first=True,
            padding_value=PAD_ID,
        ).to(DEVICE)
        translations.extend(
            translate_batch(model, source, lengths, target_vocab)
        )

    return translations


def save_checkpoint(path, model, source_vocab, target_vocab, config):
    torch.save(
        {
            "state_dict": model.state_dict(),
            "source_vocab": source_vocab.stoi,
            "target_vocab": target_vocab.stoi,
            "config": config,
        },
        path,
    )


def load_checkpoint(path):
    try:
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location=DEVICE)

    source_vocab = Vocabulary(stoi=checkpoint["source_vocab"])
    target_vocab = Vocabulary(stoi=checkpoint["target_vocab"])
    model = build_model(
        source_vocab,
        target_vocab,
        checkpoint["config"],
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, source_vocab, target_vocab, checkpoint["config"]


## 6. Load Spanish → English and train/load English → Swedish

The Spanish → English model is loaded directly from Task 4. Word2Vec initialization and the
final Task 4 configuration are reused for the new model.


In [ ]:
class TokenizedSentences:
    def __init__(self, texts):
        self.texts = texts

    def __iter__(self):
        for text in self.texts:
            yield word_tokenize(text)


def embedding_matrix(texts, vocab):
    word2vec = Word2Vec(
        TokenizedSentences(texts),
        vector_size=CONFIG["embedding_dim"],
        window=5,
        min_count=1,
        workers=max(1, min(4, torch.get_num_threads())),
        epochs=3,
        seed=SEED,
    )
    matrix = np.random.default_rng(SEED).normal(
        0,
        0.1,
        (len(vocab), CONFIG["embedding_dim"]),
    ).astype(np.float32)
    matrix[PAD_ID] = 0

    for token, index in vocab.stoi.items():
        if token in word2vec.wv:
            matrix[index] = word2vec.wv[token]
    return matrix


es_to_en_model, spanish_vocab, task4_english_vocab, task4_config = load_checkpoint(
    TASK4_CHECKPOINT
)
CONFIG = dict(task4_config)
print("Loaded Spanish → English checkpoint and configuration from Task 4.")

start = time.perf_counter()

if TASK5_CHECKPOINT.exists() and not FORCE_RETRAIN:
    en_to_sv_model, english_vocab, swedish_vocab, _ = load_checkpoint(
        TASK5_CHECKPOINT
    )
    # Rebuild loaders because checkpoint vocabularies are authoritative.
    validation_loader = make_loader(
        splits["validation"],
        english_vocab,
        swedish_vocab,
        EVALUATION_BATCH_SIZE,
        False,
    )
    test_loader = make_loader(
        splits["test"],
        english_vocab,
        swedish_vocab,
        EVALUATION_BATCH_SIZE,
        False,
    )
    history = None
    training_seconds = np.nan
    print("Loaded existing English → Swedish checkpoint.")
else:
    english_matrix = embedding_matrix(
        splits["train"]["english"],
        english_vocab,
    )
    swedish_matrix = embedding_matrix(
        splits["train"]["swedish"],
        swedish_vocab,
    )
    en_to_sv_model = build_model(
        english_vocab,
        swedish_vocab,
        CONFIG,
        english_matrix,
        swedish_matrix,
    )
    history = fit(en_to_sv_model)
    training_seconds = time.perf_counter() - start

    save_checkpoint(
        TASK5_CHECKPOINT,
        en_to_sv_model,
        english_vocab,
        swedish_vocab,
        CONFIG,
    )
    history.to_csv(
        ARTIFACT_DIR / "attention_word_en_to_sv_history.csv",
        index=False,
    )

validation_bleu, validation_chrf, _ = evaluate(
    en_to_sv_model,
    validation_loader,
    swedish_vocab,
)
test_bleu, test_chrf, test_predictions = evaluate(
    en_to_sv_model,
    test_loader,
    swedish_vocab,
)
test_predictions.to_csv(
    ARTIFACT_DIR / "attention_word_en_to_sv_predictions.csv",
    index=False,
)

results = pd.DataFrame(
    [
        {
            "direction": "English → Swedish",
            "validation_bleu": validation_bleu,
            "validation_chrf": validation_chrf,
            "test_bleu": test_bleu,
            "test_chrf": test_chrf,
            "parameters": sum(
                parameter.numel()
                for parameter in en_to_sv_model.parameters()
            ),
            "training_seconds": training_seconds,
        }
    ]
)
results.to_csv(ARTIFACT_DIR / "results.csv", index=False)
display(results)


Loaded Spanish → English checkpoint and configuration from Task 4.


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


epoch=1 train_loss=5.5474 validation_loss=4.5831


epoch=2 train_loss=4.0624 validation_loss=3.4229


epoch=3 train_loss=3.1796 validation_loss=2.9273


epoch=4 train_loss=2.7495 validation_loss=2.6908


epoch=5 train_loss=2.4949 validation_loss=2.5590


,direction,validation_bleu,validation_chrf,test_bleu,test_chrf,parameters,training_seconds
0,English → Swedish,16.337952,43.895788,16.53122,44.148645,9210464,2355.609096


## 7. Spanish → English → Swedish pivot translation

In [7]:
def translate_spanish_to_swedish(sentences):
    single_sentence = isinstance(sentences, str)
    sentences = [sentences] if single_sentence else list(sentences)

    english_pivots = translate_texts(
        es_to_en_model,
        [normalize_sentence(sentence) for sentence in sentences],
        spanish_vocab,
        task4_english_vocab,
    )
    swedish_translations = translate_texts(
        en_to_sv_model,
        english_pivots,
        english_vocab,
        swedish_vocab,
    )

    rows = pd.DataFrame(
        {
            "spanish": sentences,
            "english_pivot": english_pivots,
            "swedish": swedish_translations,
        }
    )
    return rows.iloc[0].to_dict() if single_sentence else rows


with SPANISH_PATH.open(encoding="utf-8") as file:
    candidates = [
        line.strip()
        for line in file
        if 5 <= len(word_tokenize(line.strip())) <= 12
    ]

examples = (
    pd.Series(candidates)
    .sample(n=min(5, len(candidates)), random_state=SEED)
    .tolist()
)

pivot_examples = translate_spanish_to_swedish(examples)
pivot_examples.to_csv(ARTIFACT_DIR / "pivot_examples.csv", index=False)
display(pivot_examples)


,spanish,english_pivot,swedish
0,¿por qué no armonizarlas?,why does not mean?,varför inte bara?
1,tiene sentido postergar la votación.,it has been <unk> the vote.,det har varit <unk> <unk> <unk> <unk>.
2,sería importante un compromiso del comisario s...,it would be important for the commissioner on ...,det skulle vara viktigt att kommissionsledamot...
3,actualmente está negociando con el consejo un ...,the council is currently negotiating with a fi...,rådet har nu inlett en första behandlingen.
4,por este motivo no puedo aprobar estos recortes.,that is why i cannot approve these cuts.,därför kan jag inte godkänna dessa nedskärningar.
